# Notebook 03: Classical Feature Extraction

**Purpose:** Extract classical ML features (MFCCs, spectral, temporal) from processed audio

**Key Tasks:**
1. Load processed audio segments
2. Extract MFCC features (13 coefficients × 4 stats = 52 features)
3. Extract spectral features (centroid, rolloff, bandwidth, ZCR, RMS, contrast)
4. Extract GTCC features (Gammatone Cepstral Coefficients)
5. Extract temporal features (delta and delta-delta)
6. Create three feature sets: MFCC-only, GTCC-only, Combined
7. Save unbalanced feature arrays
8. Log to MLflow

**Outputs:**
- `data/features/classical/mfcc_unbalanced_features.npy`
- `data/features/classical/gtcc_unbalanced_features.npy`
- `data/features/classical/combined_unbalanced_features.npy`
- `data/features/classical/labels.npy`

---

## 1. Import Libraries

In [16]:
import os
import sys
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Audio processing
import librosa
import librosa.feature

# Feature extraction
import spafe.features.gfcc as gfcc_module  # For GTCC
from scipy import stats
from scipy import signal

# Data manipulation
import numpy as np
import pandas as pd

# MLflow
import mlflow

# Progress tracking
from tqdm import tqdm

print('✓ Libraries imported')
print(f'librosa version: {librosa.__version__}')
print(f'numpy version: {np.__version__}')

✓ Libraries imported
librosa version: 0.10.2.post1
numpy version: 1.26.4


## 2. Project Setup

In [17]:
PROJECT_ROOT = Path('/Users/harryirving/Development/projects/ai-ml/BikeAIv5')
os.chdir(PROJECT_ROOT)

PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'universal'
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features' / 'classical'
LOGS_DIR = PROJECT_ROOT / 'logs'

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {PROJECT_ROOT}')
print(f'Processed Data: {PROCESSED_DATA_DIR}')
print(f'Features Output: {FEATURES_DIR}')

Project Root: /Users/harryirving/Development/projects/ai-ml/BikeAIv5
Processed Data: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/processed/universal
Features Output: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/classical


## 3. Setup MLflow

In [18]:
mlflow.set_tracking_uri(f'file://{LOGS_DIR / "mlruns"}')
mlflow.set_experiment('angle_grinder_pipeline')
print('✓ MLflow configured')

✓ MLflow configured


## 4. Load Manifest

In [19]:
# Load manifest from preprocessing
manifest_path = PROJECT_ROOT / 'data' / 'processed' / 'manifest.json'

if not manifest_path.exists():
    raise FileNotFoundError(f'Manifest not found: {manifest_path}. Run Notebook 02 first.')

with open(manifest_path, 'r') as f:
    manifest = json.load(f)

df_manifest = pd.DataFrame(manifest)

print(f'✓ Loaded manifest with {len(df_manifest)} segments')
print(f'\nClass distribution:')
print(df_manifest['class'].value_counts())
print(f'\nLabel distribution:')
print(df_manifest['label'].value_counts())

✓ Loaded manifest with 60324 segments

Class distribution:
class
grinder       32972
tools         19283
background     8069
Name: count, dtype: int64

Label distribution:
label
1    32972
0    27352
Name: count, dtype: int64


## 5. Define MFCC Feature Extraction

Extract 13 MFCCs with 4 statistics each (mean, std, min, max) = 52 features

In [20]:
def extract_mfcc_features(audio, sr, n_mfcc=13):
    """Extract MFCC features with statistics."""
    # Extract MFCCs
    mfccs = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=512,
        hop_length=160
    )
    
    # Calculate statistics across time for each coefficient
    features = []
    for i in range(n_mfcc):
        features.extend([
            np.mean(mfccs[i]),
            np.std(mfccs[i]),
            np.min(mfccs[i]),
            np.max(mfccs[i])
        ])
    
    return np.array(features)

print('✓ MFCC function defined')

✓ MFCC function defined


## 6. Define Spectral Feature Extraction

Extract spectral features: centroid, rolloff, bandwidth, ZCR, RMS, spectral contrast

In [21]:
def extract_spectral_features(audio, sr):
    """Extract spectral features with statistics."""
    features = []
    
    # Spectral centroid
    centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
    features.extend([np.mean(centroid), np.std(centroid)])
    
    # Spectral rolloff
    rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr, roll_percent=0.85)[0]
    features.extend([np.mean(rolloff), np.std(rolloff)])
    
    # Spectral bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr)[0]
    features.extend([np.mean(bandwidth), np.std(bandwidth)])
    
    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(audio)[0]
    features.extend([np.mean(zcr), np.std(zcr)])
    
    # RMS energy
    rms = librosa.feature.rms(y=audio)[0]
    features.extend([np.mean(rms), np.std(rms)])
    
    # Spectral contrast (7 bands × 4 stats = 28 features)
    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr, n_bands=6)
    for i in range(7):  # 6 bands + 1 = 7 bands
        features.extend([
            np.mean(contrast[i]),
            np.std(contrast[i]),
            np.min(contrast[i]),
            np.max(contrast[i])
        ])
    
    # Spectral skewness
    S = np.abs(librosa.stft(audio))
    skewness = stats.skew(S, axis=-1)
    features.extend([np.mean(skewness), np.std(skewness)])
    
    return np.array(features)

print('✓ Spectral features function defined')

✓ Spectral features function defined


## 7. Define GTCC Feature Extraction

Gammatone Cepstral Coefficients - alternative to MFCCs

In [22]:
def extract_gtcc_features(audio, sr, n_gtcc=13):
    """Extract GTCC features with statistics."""
    try:
        # Extract GTCCs using spafe
        gtccs = gfcc_module.gfcc(
            sig=audio,
            fs=sr,
            num_ceps=n_gtcc,
            nfilts=100,
            low_freq=50,
            high_freq=8000
        )
        
        # Transpose to get (n_coeffs, n_frames)
        gtccs = gtccs.T
        
        # Calculate statistics
        features = []
        for i in range(n_gtcc):
            features.extend([
                np.mean(gtccs[i]),
                np.std(gtccs[i]),
                np.min(gtccs[i]),
                np.max(gtccs[i])
            ])
        
        return np.array(features)
    
    except Exception as e:
        # If GTCC fails, return zeros
        return np.zeros(n_gtcc * 4)

print('✓ GTCC function defined')

✓ GTCC function defined


## 8. Define Temporal Feature Extraction

Delta and delta-delta MFCCs and GTCCs

In [23]:
def extract_temporal_features(audio, sr, n_mfcc=13):
    """Extract delta and delta-delta MFCC features."""
    # Extract MFCCs
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc, n_fft=512, hop_length=160)
    
    # Calculate deltas
    delta_mfccs = librosa.feature.delta(mfccs, order=1)
    delta2_mfccs = librosa.feature.delta(mfccs, order=2)
    
    features = []
    
    # Delta stats
    for i in range(n_mfcc):
        features.extend([
            np.mean(delta_mfccs[i]),
            np.std(delta_mfccs[i]),
            np.min(delta_mfccs[i]),
            np.max(delta_mfccs[i])
        ])
    
    # Delta-delta stats
    for i in range(n_mfcc):
        features.extend([
            np.mean(delta2_mfccs[i]),
            np.std(delta2_mfccs[i]),
            np.min(delta2_mfccs[i]),
            np.max(delta2_mfccs[i])
        ])
    
    return np.array(features)

print('✓ Temporal features function defined')

✓ Temporal features function defined


## 9. Define MSES (Multi-band Spectral Entropy)

Additional spectral feature for analysis

In [24]:
def extract_mses_features(audio, sr, n_bands=6):
    """Extract multi-band spectral entropy."""
    # Compute spectrogram
    S = np.abs(librosa.stft(audio))
    
    # Divide into frequency bands
    band_size = S.shape[0] // n_bands
    
    features = []
    for i in range(n_bands):
        start = i * band_size
        end = (i + 1) * band_size if i < n_bands - 1 else S.shape[0]
        band = S[start:end, :]
        
        # Calculate entropy for this band
        band_sum = np.sum(band, axis=0) + 1e-10  # Avoid log(0)
        normalized = band / band_sum
        entropy = -np.sum(normalized * np.log2(normalized + 1e-10), axis=0)
        
        features.extend([np.mean(entropy), np.std(entropy)])
    
    return np.array(features)

print('✓ MSES function defined')

✓ MSES function defined


## 10. Define Combined Feature Extraction

Wrapper function to extract all features from a single audio file

In [25]:
def extract_all_features(filepath, sr=16000):
    """Extract all classical features from an audio file."""
    try:
        # Load audio
        audio, _ = librosa.load(filepath, sr=sr)
        
        # Extract features
        mfcc_feats = extract_mfcc_features(audio, sr)  # 52 features
        spectral_feats = extract_spectral_features(audio, sr)  # ~40 features
        gtcc_feats = extract_gtcc_features(audio, sr)  # 52 features
        temporal_feats = extract_temporal_features(audio, sr)  # 104 features (delta + delta-delta)
        mses_feats = extract_mses_features(audio, sr)  # 12 features (6 bands × 2 stats)
        
        # Create feature sets
        mfcc_set = np.concatenate([mfcc_feats, spectral_feats, temporal_feats, mses_feats])
        gtcc_set = np.concatenate([gtcc_feats, spectral_feats, temporal_feats, mses_feats])
        combined_set = np.concatenate([mfcc_feats, gtcc_feats, spectral_feats, temporal_feats, mses_feats])
        
        return {
            'mfcc': mfcc_set,
            'gtcc': gtcc_set,
            'combined': combined_set,
            'status': 'success'
        }
    
    except Exception as e:
        return {
            'status': 'failed',
            'error': str(e)
        }

print('✓ Combined extraction function defined')

✓ Combined extraction function defined


## 11. Extract Features from All Segments

Process all audio segments and create feature matrices

In [26]:
print('Extracting features from all segments...')
print('='*70)

mfcc_features_list = []
gtcc_features_list = []
combined_features_list = []
labels_list = []
failed_count = 0

for idx, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc='Extracting'):
    filepath = PROJECT_ROOT / row['filepath']
    
    if not filepath.exists():
        failed_count += 1
        continue
    
    result = extract_all_features(filepath, sr=16000)
    
    if result['status'] == 'success':
        mfcc_features_list.append(result['mfcc'])
        gtcc_features_list.append(result['gtcc'])
        combined_features_list.append(result['combined'])
        labels_list.append(row['label'])
    else:
        failed_count += 1

print(f'\n✓ Extracted features from {len(mfcc_features_list)} segments')
if failed_count > 0:
    print(f'⚠️  Failed to extract features from {failed_count} segments')

Extracting features from all segments...


Extracting:   6%|▌         | 3551/60324 [01:11<19:05, 49.57it/s]


KeyboardInterrupt: 

In [ ]:
print(combined_features_list)

NameError: name 'combined_features_list' is not defined

## 12. Convert to NumPy Arrays

In [ ]:
# Convert lists to arrays
X_mfcc = np.array(mfcc_features_list)
X_gtcc = np.array(gtcc_features_list)
X_combined = np.array(combined_features_list)
y = np.array(labels_list)

print('Feature Array Shapes:')
print('='*70)
print(f'MFCC features:     {X_mfcc.shape}')
print(f'GTCC features:     {X_gtcc.shape}')
print(f'Combined features: {X_combined.shape}')
print(f'Labels:            {y.shape}')
print('='*70)

print(f'\nLabel distribution:')
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    label_name = 'grinder' if label == 1 else 'non-grinder'
    print(f'  {label_name}: {count} samples ({count/len(y)*100:.1f}%)')

NameError: name 'np' is not defined

## 13. Save Feature Arrays

In [ ]:
# Save feature arrays
np.save(FEATURES_DIR / 'mfcc_unbalanced_features.npy', X_mfcc)
print(f'✓ Saved: {FEATURES_DIR / "mfcc_unbalanced_features.npy"}')

np.save(FEATURES_DIR / 'gtcc_unbalanced_features.npy', X_gtcc)
print(f'✓ Saved: {FEATURES_DIR / "gtcc_unbalanced_features.npy"}')

np.save(FEATURES_DIR / 'combined_unbalanced_features.npy', X_combined)
print(f'✓ Saved: {FEATURES_DIR / "combined_unbalanced_features.npy"}')

np.save(FEATURES_DIR / 'labels.npy', y)
print(f'✓ Saved: {FEATURES_DIR / "labels.npy"}')

# Save feature metadata
metadata = {
    'timestamp': datetime.now().isoformat(),
    'total_samples': len(y),
    'mfcc_features_shape': X_mfcc.shape,
    'gtcc_features_shape': X_gtcc.shape,
    'combined_features_shape': X_combined.shape,
    'label_distribution': {int(k): int(v) for k, v in zip(*np.unique(y, return_counts=True))},
    'feature_descriptions': {
        'mfcc': 'MFCC (52) + Spectral (~40) + Temporal (104) + MSES (12)',
        'gtcc': 'GTCC (52) + Spectral (~40) + Temporal (104) + MSES (12)',
        'combined': 'All MFCC + GTCC + Spectral + Temporal + MSES features'
    }
}

with open(FEATURES_DIR / 'feature_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f'✓ Saved: {FEATURES_DIR / "feature_metadata.json"}')

✓ Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/classical/mfcc_unbalanced_features.npy
✓ Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/classical/gtcc_unbalanced_features.npy
✓ Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/classical/combined_unbalanced_features.npy
✓ Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/classical/labels.npy
✓ Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/classical/feature_metadata.json


## 14. Log to MLflow

In [ ]:
with mlflow.start_run(run_name='03_classical_feature_extraction'):
    # Log metrics
    mlflow.log_metric('total_samples', len(y))
    mlflow.log_metric('mfcc_feature_dim', X_mfcc.shape[1])
    mlflow.log_metric('gtcc_feature_dim', X_gtcc.shape[1])
    mlflow.log_metric('combined_feature_dim', X_combined.shape[1])
    mlflow.log_metric('grinder_samples', int(np.sum(y == 1)))
    mlflow.log_metric('non_grinder_samples', int(np.sum(y == 0)))
    mlflow.log_metric('failed_extractions', failed_count)
    
    # Log parameters
    mlflow.log_param('n_mfcc', 13)
    mlflow.log_param('n_gtcc', 13)
    mlflow.log_param('n_spectral_bands', 6)
    mlflow.log_param('feature_extraction_sr', 44100)
    
    # Log tags
    mlflow.set_tags({
        'stage': 'feature_extraction',
        'notebook': '03',
        'feature_type': 'classical',
        'timestamp': datetime.now().isoformat()
    })
    
    # Log artifacts
    mlflow.log_artifact(str(FEATURES_DIR / 'feature_metadata.json'))
    
    print('\n✓ Logged to MLflow')


✓ Logged to MLflow


## 15. Summary and Next Steps

In [ ]:
print('\n' + '='*70)
print('CLASSICAL FEATURE EXTRACTION COMPLETE')
print('='*70)
print(f'\n✓ Extracted features from {len(y)} segments')
print(f'\nFeature Set Dimensions:')
print(f'  MFCC-based:    {X_mfcc.shape[1]} features')
print(f'  GTCC-based:    {X_gtcc.shape[1]} features')
print(f'  Combined:      {X_combined.shape[1]} features')
print(f'\nClass Balance (unbalanced):')
for label, count in zip(*np.unique(y, return_counts=True)):
    label_name = 'grinder' if label == 1 else 'non-grinder'
    print(f'  {label_name}: {count} samples')
print(f'\nSaved to: {FEATURES_DIR}')
print('\nNext Steps:')
print('1. Proceed to training:')
print('   → 05_classical_training_and_tuning.ipynb')
print('     (Will handle data splitting and balancing)')
print('\n2. Or extract neural features in parallel:')
print('   → 04_neural_feature_extraction.ipynb')
print('\n' + '='*70)
print('\n✓ Notebook 03 Complete!')


CLASSICAL FEATURE EXTRACTION COMPLETE

✓ Extracted features from 60348 segments

Feature Set Dimensions:
  MFCC-based:    208 features
  GTCC-based:    208 features
  Combined:      260 features

Class Balance (unbalanced):
  non-grinder: 27410 samples
  grinder: 32938 samples

Saved to: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/classical

Next Steps:
1. Proceed to training:
   → 05_classical_training_and_tuning.ipynb
     (Will handle data splitting and balancing)

2. Or extract neural features in parallel:
   → 04_neural_feature_extraction.ipynb


✓ Notebook 03 Complete!
